In [16]:
import pandas as pd
import numpy as np

# Load the original raw dataset
df = pd.read_csv('../data/laptop_price.csv', encoding='latin1')
df.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


In [17]:
df.shape

(1303, 13)

In [18]:
df = df.drop(columns=['laptop_ID', 'Product'])
print('Columns after dropping useless ones:')
print(df.columns.tolist())

Columns after dropping useless ones:
['Company', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'Price_euros']


In [19]:
print('BEFORE:', df['Ram'].unique())
df['Ram'] = df['Ram'].str.replace('GB', '', regex=False).astype(int)
print('AFTER :', sorted(df['Ram'].unique()))

BEFORE: <StringArray>
['8GB', '16GB', '4GB', '2GB', '12GB', '6GB', '32GB', '24GB', '64GB']
Length: 9, dtype: str
AFTER : [np.int64(2), np.int64(4), np.int64(6), np.int64(8), np.int64(12), np.int64(16), np.int64(24), np.int64(32), np.int64(64)]


In [20]:
print('BEFORE:', df['Weight'].unique()[:5])
df['Weight'] = df['Weight'].str.replace('kg', '', regex=False).astype(float)
print('AFTER :', sorted(df['Weight'].unique())[:5])

BEFORE: <StringArray>
['1.37kg', '1.34kg', '1.86kg', '1.83kg', '2.1kg']
Length: 5, dtype: str
AFTER : [np.float64(0.69), np.float64(0.81), np.float64(0.91), np.float64(0.92), np.float64(0.97)]


In [21]:
print('BEFORE :', df['Cpu'].unique()[:3])
df['Cpu_Brand'] = df['Cpu'].apply(lambda x: x.split()[0])

print('AFTER — unique brands:', df['Cpu_Brand'].unique())

BEFORE : <StringArray>
['Intel Core i5 2.3GHz', 'Intel Core i5 1.8GHz', 'Intel Core i5 7200U 2.5GHz']
Length: 3, dtype: str
AFTER — unique brands: <StringArray>
['Intel', 'AMD', 'Samsung']
Length: 3, dtype: str


In [22]:
print('BEFORE :', df['Gpu'].unique()[:3])
df['Gpu_Brand'] = df['Gpu'].apply(lambda x: x.split()[0])

print('AFTER — unique brands:', df['Gpu_Brand'].unique())

BEFORE : <StringArray>
['Intel Iris Plus Graphics 640',       'Intel HD Graphics 6000',
        'Intel HD Graphics 620']
Length: 3, dtype: str
AFTER — unique brands: <StringArray>
['Intel', 'AMD', 'Nvidia', 'ARM']
Length: 4, dtype: str


In [23]:
print('BEFORE :', df['Memory'].unique()[:6])

# Check if the word 'SSD' is anywhere in the Memory column
df['Has_SSD'] = df['Memory'].apply(lambda x: 1 if 'SSD' in x else 0)
df['Has_HDD'] = df['Memory'].apply(lambda x: 1 if 'HDD' in x else 0)

print('\nHas_SSD value counts:')
df['Has_SSD'].value_counts()
print('\nHas_HDD value counts:')
df['Has_HDD'].value_counts()

BEFORE : <StringArray>
[          '128GB SSD', '128GB Flash Storage',           '256GB SSD',
           '512GB SSD',           '500GB HDD', '256GB Flash Storage']
Length: 6, dtype: str

Has_SSD value counts:

Has_HDD value counts:


Has_HDD
0    727
1    576
Name: count, dtype: int64

In [25]:
print('BEFORE :', df['ScreenResolution'].unique()[:4])

def extract_width(res_str):
    last_part = res_str.split()[-1]   
    width = last_part.split('x')[0]   
    return int(width)

def extract_height(res_str):
    last_part = res_str.split()[-1]
    height = last_part.split('x')[1]  
    return int(height)

df['Screen_Width']  = df['ScreenResolution'].apply(extract_width)
df['Screen_Height'] = df['ScreenResolution'].apply(extract_height)

print('\nAFTER — Screen_Width  sample:', df['Screen_Width'].unique())
print('AFTER — Screen_Height sample:', df['Screen_Height'].unique())

BEFORE : <StringArray>
['IPS Panel Retina Display 2560x1600',                           '1440x900',
                  'Full HD 1920x1080', 'IPS Panel Retina Display 2880x1800']
Length: 4, dtype: str

AFTER — Screen_Width  sample: [2560 1440 1920 2880 1366 2304 3200 2256 3840 2160 1600 2736 2400]
AFTER — Screen_Height sample: [1600  900 1080 1800  768 1440 1200 1504 2160 1824]


In [26]:
cols_to_drop = ['Cpu', 'Gpu', 'Memory', 'ScreenResolution']
df = df.drop(columns=cols_to_drop)

print('Columns after dropping messy originals:')
print(df.columns.tolist())

Columns after dropping messy originals:
['Company', 'TypeName', 'Inches', 'Ram', 'OpSys', 'Weight', 'Price_euros', 'Cpu_Brand', 'Gpu_Brand', 'Has_SSD', 'Has_HDD', 'Screen_Width', 'Screen_Height']


In [27]:
print('Shape BEFORE encoding:', df.shape)

# List the text columns that need to be encoded
categorical_columns = ['Company', 'TypeName', 'OpSys', 'Cpu_Brand', 'Gpu_Brand']

# pd.get_dummies() does all the one-hot encoding automatically
df = pd.get_dummies(df, columns=categorical_columns)

print('Shape AFTER  encoding:', df.shape)
print(f'\nWe now have {df.shape[1]} columns (more columns, but all numbers!)')

Shape BEFORE encoding: (1303, 13)
Shape AFTER  encoding: (1303, 49)

We now have 49 columns (more columns, but all numbers!)


In [28]:
print('First 5 rows of cleaned data:')
df.head()

First 5 rows of cleaned data:


,Inches,Ram,Weight,Price_euros,Has_SSD,Has_HDD,Screen_Width,Screen_Height,Company_Acer,Company_Apple,...,OpSys_Windows 10 S,OpSys_Windows 7,OpSys_macOS,Cpu_Brand_AMD,Cpu_Brand_Intel,Cpu_Brand_Samsung,Gpu_Brand_AMD,Gpu_Brand_ARM,Gpu_Brand_Intel,Gpu_Brand_Nvidia
0,13.3,8,1.37,1339.69,1,0,2560,1600,False,True,...,False,False,True,False,True,False,False,False,True,False
1,13.3,8,1.34,898.94,0,0,1440,900,False,True,...,False,False,True,False,True,False,False,False,True,False
2,15.6,8,1.86,575.00,1,0,1920,1080,False,False,...,False,False,False,False,True,False,False,False,True,False
3,15.4,16,1.83,2537.45,1,0,2880,1800,False,True,...,False,False,True,False,True,False,True,False,False,False
4,13.3,8,1.37,1803.60,1,0,2560,1600,False,True,...,False,False,True,False,True,False,False,False,True,False


In [29]:
print('Data types (should all be numeric now):')
print(df.dtypes.value_counts())

print(f'\nTotal missing values: {df.isnull().sum().sum()}')

# Check that Price_euros is still in the dataset
print(f'\nPrice_euros still present: {"Price_euros" in df.columns}')

Data types (should all be numeric now):
bool       41
int64       5
float64     3
Name: count, dtype: int64

Total missing values: 0

Price_euros still present: True


In [30]:
df.to_csv('../data/laptop_price_cleaned.csv', index=False)

print('Cleaned dataset saved to: ../data/laptop_price_cleaned.csv')
print(f'Final shape: {df.shape}')

Cleaned dataset saved to: ../data/laptop_price_cleaned.csv
Final shape: (1303, 49)
